In [1]:
import pandas as pd
import os
import requests
import zipfile
import io
import matplotlib.pyplot as plt
import seaborn as sns
import requests
import warnings
import numpy as np
warnings.simplefilter(action='ignore')
Dir_github = r'X:\PROJECTS\ABC@UofSC\3-Experiments\NSF_valence\github'
# Dir_segments = os.path.join(Dir_github,'data','segmentation')
# Dir_gentle = os.path.join(Dir_github,'data','timestamps','timestamps_word')
repo = 'ABC_valence'


In [ ]:
def get_qualtrics_survey(dir_save_survey, survey_id):
    """ automatically query the qualtrics survey data
    guide https://community.alteryx.com/t5/Alteryx-Designer-Discussions/Python-Tool-Downloading-Qualtrics-Survey-Data-using-Python-API/td-p/304898 """

    # Setting user Parameters
    api_token = "QUahqaqAx3GWF0pswKjbYYR37cil6xd0vN2zpB85" # your API ID
    file_format = "csv"
    data_center = 'uofsc.sjc1' # "<Organization ID>.<Datacenter ID>"

    # Setting static parameters
    request_check_progress = 0
    progress_status = "in progress"
    base_url = "https://{0}.qualtrics.com/API/v3/responseexports/".format(data_center)
    headers = {
        "content-type": "application/json",
        "x-api-token": api_token,
    }

    # Step 1: Creating Data Export
    download_request_url = base_url
    download_request_payload = '{"format":"' + file_format + '","surveyId":"' + survey_id + '"}' # you can set useLabels:True to get responses in text format
    download_request_response = requests.request("POST", download_request_url, data=download_request_payload, headers=headers)
    progress_id = download_request_response.json()["result"]["id"]
    # print(download_request_response.text)

    # Step 2: Checking on Data Export Progress and waiting until export is ready
    while request_check_progress < 100 and progress_status != "complete":
        request_check_url = base_url + progress_id
        request_check_response = requests.request("GET", request_check_url, headers=headers)
        request_check_progress = request_check_response.json()["result"]["percentComplete"]

    # Step 3: Downloading file
    request_download_url = base_url + progress_id + '/file'
    request_download = requests.request("GET", request_download_url, headers=headers, stream=True)

    # Step 4: Unzipping the file
    zipfile.ZipFile(io.BytesIO(request_download.content)).extractall(dir_save_survey)
    print('Downloaded qualtrics survey',survey_id)

def get_allsurveys():
        
    # Replace these with your actual Qualtrics API token and data center ID
    api_token = "QUahqaqAx3GWF0pswKjbYYR37cil6xd0vN2zpB85" # your API ID
    data_center = 'uofsc.sjc1' # "<Organization ID>.<Datacenter ID>"

    # Qualtrics API endpoint to list surveys
    url = f"https://{data_center}.qualtrics.com/API/v3/surveys"

    # HTTP headers for authentication
    headers = {
        "Content-Type": "application/json",
        "X-API-TOKEN": api_token,
    }

    try:
        # Send GET request to Qualtrics API
        response = requests.get(url, headers=headers)
        response.raise_for_status()  # Raise an error for HTTP codes 4xx or 5xx
        
        # Parse the JSON response
        data = response.json()
        surveys = data.get("result", {}).get("elements", [])
        
    except requests.exceptions.RequestException as e:
        print(f"Error: {e}")

    return(surveys)

Download data

In [17]:
df_surveys = pd.DataFrame(get_allsurveys())
df_surveys = df_surveys.loc[df_surveys['name'].str.contains('ABCNaturalisticNarrativeAudio')]
df_surveys = df_surveys.loc[~df_surveys['name'].str.contains('template')]
df_surveys

,id,name,ownerId,lastModified,creationDate,isActive
3,SV_0Cl7sWMHjgiHYyy,ABCNaturalisticNarrativeAudio_Jacob_FireSelected,UR_5uvSniYAMbURnkV,2024-03-21T20:41:44Z,2024-03-21T20:40:58Z,True
17,SV_1TzZqiVRKdHis1E,ABCNaturalisticNarrativeAudio_WavenetB_Bullies...,UR_5uvSniYAMbURnkV,2024-02-22T02:39:14Z,2024-02-22T02:36:23Z,True
18,SV_1yHZJgIi7N5cCBU,ABCNaturalisticNarrativeAudio_Jack_Elvis_old1008,UR_5uvSniYAMbURnkV,2024-03-01T02:30:40Z,2024-03-01T02:30:40Z,False
24,SV_2adux484NpUGrHM,ABCNaturalisticNarrativeAudio_Jacob_UserFriendly,UR_5uvSniYAMbURnkV,2024-03-21T16:57:41Z,2024-03-21T16:56:11Z,True
30,SV_3aBgaWwVUX3h52e,ABCNaturalisticNarrativeAudio_Jacob_TaleofTwoC...,UR_5uvSniYAMbURnkV,2024-03-26T15:47:49Z,2024-03-26T15:47:10Z,True
36,SV_3ggk7pEPVPtE4hU,ABCNaturalisticNarrativeAudio_Brandi_BradfordBull,UR_5uvSniYAMbURnkV,2024-02-20T22:27:58Z,2024-02-20T17:59:27Z,True
38,SV_3PJ637jegsjFh6S,ABCNaturalisticNarrativeAudio_Brandi_TaleofTwo...,UR_5uvSniYAMbURnkV,2024-02-29T22:47:03Z,2024-02-29T22:46:07Z,True
41,SV_3rdmbC8sgZwzgN0,ABCNaturalisticNarrativeAudio_Jacob_HowCouldYou,UR_5uvSniYAMbURnkV,2024-03-26T15:49:51Z,2024-03-26T15:49:19Z,True
42,SV_3s0FBcBNVE8Nx8G,ABCNaturalisticNarrativeAudio_Jack_BiggestWastes,UR_5uvSniYAMbURnkV,2024-02-20T19:27:25Z,2024-02-20T19:25:02Z,True
46,SV_3XjHXIp9HONDIV0,ABCNaturalisticNarrativeAudio_Allison_Metaphor...,UR_5uvSniYAMbURnkV,2024-03-01T02:24:54Z,2024-03-01T01:58:46Z,True


In [19]:
pipeline = 'Rutvik_fMRITMS'
Dir_output = os.path.join(Dir_github,'data','qualtrics',pipeline)
if not os.path.exists(Dir_output):
    os.makedirs(Dir_output)

for surveyID in df_surveys['id'].unique():  
    # surveyID = 'SV_0Cl7sWMHjgiHYyy'
    get_qualtrics_survey(Dir_output, surveyID)

Downloaded qualtrics survey SV_0Cl7sWMHjgiHYyy
Downloaded qualtrics survey SV_1TzZqiVRKdHis1E
Downloaded qualtrics survey SV_1yHZJgIi7N5cCBU
Downloaded qualtrics survey SV_2adux484NpUGrHM
Downloaded qualtrics survey SV_3aBgaWwVUX3h52e
Downloaded qualtrics survey SV_3ggk7pEPVPtE4hU
Downloaded qualtrics survey SV_3PJ637jegsjFh6S
Downloaded qualtrics survey SV_3rdmbC8sgZwzgN0
Downloaded qualtrics survey SV_3s0FBcBNVE8Nx8G
Downloaded qualtrics survey SV_3XjHXIp9HONDIV0
Downloaded qualtrics survey SV_4Jb7kK94OtjuWhM
Downloaded qualtrics survey SV_54NOgaxC2BqaEPI
Downloaded qualtrics survey SV_5i17hqxDUOwCNqS
Downloaded qualtrics survey SV_5nnZQN6s9kVz7gy
Downloaded qualtrics survey SV_73zrmlbVTEScNym
Downloaded qualtrics survey SV_7UTyoZqdbg0AjJ4
Downloaded qualtrics survey SV_82DCHtgbNOCzsl8
Downloaded qualtrics survey SV_8Cgc15tgRfQdrrE
Downloaded qualtrics survey SV_8lkJnlEKEb929xk


Import data

In [ ]:
pipeline = 'Rutvik_fMRITMS'
Dir_qualtrics = os.path.join(Dir_github,'data','qualtrics',pipeline)
List_time = []
dfs_raw = []
for file in os.listdir(Dir_qualtrics):
    # if 'Jack_Biggest' in file:
    _,audio = file.split('.')[0].split('_',1)
    data_raw = pd.read_csv(os.path.join(Dir_qualtrics,file))
    data = data_raw.loc[data_raw['Finished']=='1'].copy() # only get the participants who have finished the entire experiment
    data['EndDate'] = pd.to_datetime(data['EndDate'])
    data['StartDate'] = pd.to_datetime(data['StartDate'])
    data['completiontime'] = data['EndDate'] - data['StartDate']
    data['completiontime'] = data['completiontime'].dt.total_seconds() / 60

    # for i, row in data.iterrows():
    #     # print(audio,i,row['completiontime'])
    #     List_time.append(row['completiontime'])
        
    for i, row in data.iterrows():
        List_vars_V = [x for x in data.columns if 'Q11_1_x' in x]
        List_vars_A = [x for x in data.columns if 'Q11_1_y' in x]

        df_tmp = pd.DataFrame({'ResponseID':row['ResponseID'],'subid':row['ID'],'audio':audio,'age':row['Q1'],'gender':row['Q3'],'completiontime':row['completiontime'],
                        'A':row[List_vars_A].values,'V':row[List_vars_V].values,'vivid':row['Q22_4']})
        df_tmp['idx'] = range(df_tmp.shape[0])
        dfs_raw.append(df_tmp)
    
        # break
dfs_raw = pd.concat(dfs_raw,ignore_index=True)
dfs_raw['A'] = dfs_raw['A'].astype(float)
dfs_raw['V'] = dfs_raw['V'].astype(float)
dfs_raw

,ResponseID,subid,audio,age,gender,completiontime,A,V,vivid,idx
0,R_6LmsnnJ69BYT57j,1010,Allison_MetaphorsPronouns,18,female,12.233333,165,128,4,0
1,R_6LmsnnJ69BYT57j,1010,Allison_MetaphorsPronouns,18,female,12.233333,112,105,4,1
2,R_6LmsnnJ69BYT57j,1010,Allison_MetaphorsPronouns,18,female,12.233333,113,92,4,2
3,R_6LmsnnJ69BYT57j,1010,Allison_MetaphorsPronouns,18,female,12.233333,100,92,4,3
4,R_6LmsnnJ69BYT57j,1010,Allison_MetaphorsPronouns,18,female,12.233333,132,90,4,4
...,...,...,...,...,...,...,...,...,...,...
1544,R_7EtqxLthEce2z2x,1014,Jacob_UserFriendly,39,Male,22.400000,317,183,3,83
1545,R_7EtqxLthEce2z2x,1014,Jacob_UserFriendly,39,Male,22.400000,333,293,3,84
1546,R_7EtqxLthEce2z2x,1014,Jacob_UserFriendly,39,Male,22.400000,359,242,3,85
1547,R_7EtqxLthEce2z2x,1014,Jacob_UserFriendly,39,Male,22.400000,314,221,3,86
